<a href="https://colab.research.google.com/github/mamontoff2018-lgtm/BerezinaAlg/blob/main/practices/LR3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ЛАБОРАТОРНАЯ РАБОТА №3. Деревья решений (Decision Trees)

## Задания


In [4]:
import numpy as np
from collections import Counter

class Node:

    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf_node(self):
        return self.value is not None


class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None

    def fit(self, X, y):
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        if n_samples == 0:
            return Node(value=-1)

        n_labels = len(np.unique(y))

        # Критерии остановки
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # Выбираем случайный порядок признаков (для разнообразия деревьев)
        feat_idxs = np.random.choice(n_feats, n_feats, replace=False)
        best_feat, best_thresh = self._best_split(X, y, feat_idxs)

        # Рекурсивное построение поддеревьев
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)

        return Node(best_feat, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)

            for thr in thresholds:
                gain = self._information_gain(y, X_column, thr)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold

    def _information_gain(self, y, X_column, threshold):
        parent_entropy = self._entropy(y)

        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._entropy(y[left_idxs]), self._entropy(y[right_idxs])
        child_entropy = (n_l / n) * e_l + (n_r / n) * e_r

        return parent_entropy - child_entropy

    def _entropy(self, y):
        hist = np.bincount(y)
        ps = hist / len(y)
        return -np.sum([p * np.log2(p) for p in ps if p > 0])

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

    def print_tree(self, node=None, depth=0):
        if node is None:
            node = self.root
        if node.is_leaf_node():
            print(f"{'  ' * depth}Результат: Класс {node.value}")
            return
        print(f"{'  ' * depth}[Признак {node.feature} <= {node.threshold:.2f}]")
        print(f"{'  ' * depth} Лево:")
        self.print_tree(node.left, depth + 1)
        print(f"{'  ' * depth} Право:")
        self.print_tree(node.right, depth + 1)

In [3]:
from sklearn import datasets
from sklearn.model_selection import train_test_split

data = datasets.load_iris()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

clf = DecisionTree(max_depth=3, min_samples_split=2)
clf.fit(X_train, y_train)

predictions = clf.predict(X_test)

def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

acc = accuracy(y_test, predictions)
print(f"Точность модели на тестовой выборке: {acc * 100:.2f}%")

print("\nРеальные метки:    ", y_test)
print("Предсказанные:     ", predictions)

Точность модели на тестовой выборке: 96.67%

Реальные метки:     [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Предсказанные:      [1 0 2 1 2 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]


## Контрольные вопросы

### 1. Почему используется логарифм по основанию 2 в формуле энтропии?

  Логарифм с основанием 2 даёт единицу измерения «бит» — количество информации, необходимое для однозначного определения класса случайного объекта (задавая вопросы с ответом «да/нет»).  
  Например, если в узле поровну объектов двух классов, энтропия равна `1 бит` (полная неопределённость). С любым другим основанием суть не изменится, но битовая интерпретация наиболее удобна в теории информации и машинном обучении.

### 2. Что произойдёт, если в признаке очень много уникальных значений? Почему информационный выигрыш может «ошибочно» посчитать такой признак лучшим?

Признак с большим количеством уникальных значений (вплоть до уникальных записей для каждого объекта) позволяет разбить данные на множество мелких подгрупп, в каждой из которых энтропия будет нулевой (все объекты одного класса).  
Тогда **Information Gain становится максимальным**, но это не настоящее знание — дерево просто «запоминает» обучающую выборку, переобучается и теряет обобщающую способность. В реальности такой признак часто является идентификатором (например, номер строки) и не несёт предсказательной ценности.

### 3. В чём преимущество дерева решений перед kNN с точки зрения интерпретируемости результата человеком?

Дерево решений формирует явную иерархию **правил «если ... то ...»**, которую можно прочитать, визуализировать и объяснить.  
Путь от корня до листа — это последовательность сравнений признаков с порогами, понятная даже неспециалисту.  
kNN же — «чёрный ящик»: решение принимается на основе расстояния до соседей, и объяснить, почему именно этот класс, можно только перечислив соседей, без причинно-следственной логики. Дерево даёт **интерпретируемую модель** в отличие от запоминающего kNN.

### 4. Когда следует прекратить рекурсию при построении дерева? (минимум два условия)

В нашем коде явно заданы следующие критерии остановки:

1. **Достигнута максимальная глубина** (`depth >= self.max_depth`).  
   Это предотвращает бесконечное дробление и ограничивает сложность модели.

2. **Число образцов в узле меньше минимально допустимого для разбиения** (`n_samples < self.min_samples_split`).  
   Слишком маленький узел не даёт статистически надёжного разбиения.

Дополнительно, рекурсия прекращается, если:

- все объекты в узле принадлежат **одному классу** (`n_labels == 1`), то есть узел уже «чистый» и не требует дальнейшего деления.
